# DefectosCafeVerde grouped — RATF1 seed 42

Fresh 50-epoch Redundancy-Aware Selective Texture Fusion from official YOLO26n. RGB and the native box path remain intact; test is never extracted.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import csv, hashlib, importlib, json, os, shutil, subprocess, sys, tarfile, time
ARM='RATF1'; BRANCH='codex/defectoscafeverde-ratf1'
WORK=Path('/content'); REPO=WORK/'coffee-bean-detection'
ARCHIVE_NAME='defectoscafeverde-grouped-physical-v1.tar'
EXPECTED_ARCHIVE_BYTES=686474752
EXPECTED_ARCHIVE_SHA='53fb2233f1f0d1c77cb24eca2d720f86e0a16835b8a69f4e8f3176fae1aacef2'
def sha256(path):
    h=hashlib.sha256()
    with Path(path).open('rb') as stream:
        for block in iter(lambda:stream.read(8*1024*1024),b''): h.update(block)
    return h.hexdigest()
drive_roots=[root for root in [Path('/content/drive/MyDrive'),*Path('/content/drive/.shortcut-targets-by-id').glob('*')] if root.exists()]
d0_relative=Path('experiments/defectoscafeverde-cwcf-direct-v1/val_reports/D0DIRECT_seed42_result.json')
d0_candidates=sorted({p for root in drive_roots for p in root.rglob(d0_relative.name) if p.as_posix().endswith(d0_relative.as_posix())})
if len(d0_candidates)!=1: raise FileNotFoundError(f'Harus ada tepat satu D0 reference; ditemukan {d0_candidates}')
D0=d0_candidates[0]; PROJECT=D0.parents[3]
archives=sorted({p for root in drive_roots for p in root.rglob(ARCHIVE_NAME) if p.is_file()})
valid_archives=[p for p in archives if p.stat().st_size==EXPECTED_ARCHIVE_BYTES and sha256(p)==EXPECTED_ARCHIVE_SHA]
complete_parts=[]
for scheme,count in (('part16',41),('chunk',11),('part',2)):
    by_parent={}
    for root in drive_roots:
        for p in root.rglob(f'{ARCHIVE_NAME}.{scheme}-*'):
            if p.is_file(): by_parent.setdefault(str(p.parent),{})[p.name]=p
    expected=[f'{ARCHIVE_NAME}.{scheme}-{i:03d}' for i in range(count)]
    for parent,rows in by_parent.items():
        if sorted(rows)==expected and sum(rows[n].stat().st_size for n in expected)==EXPECTED_ARCHIVE_BYTES: complete_parts.append((scheme,parent,[rows[n] for n in expected]))
if valid_archives:
    ARCHIVE=valid_archives[0]; ARCHIVE_SOURCE='single archive'
elif complete_parts:
    scheme,parent,parts=complete_parts[0]; ARCHIVE=WORK/ARCHIVE_NAME; ARCHIVE_SOURCE=f'{len(parts)} {scheme} parts dari {parent}'
    with ARCHIVE.open('wb') as target:
        for part in parts:
            with part.open('rb') as source: shutil.copyfileobj(source,target,length=8*1024*1024)
    if sha256(ARCHIVE)!=EXPECTED_ARCHIVE_SHA: raise RuntimeError('SHA hasil gabung tidak cocok')
else:
    raise FileNotFoundError('Bundle Defectos tidak terlihat dari akun ini. Tambahkan shortcut folder bersama ke My Drive atau attach bundle.')
print('PROJECT:',PROJECT); print('ARCHIVE VALID:',ARCHIVE,'| SOURCE:',ARCHIVE_SOURCE)

In [ ]:
os.chdir(WORK)
if REPO.exists(): shutil.rmtree(REPO)
subprocess.run(['git','clone','--depth','1','--branch',BRANCH,'https://github.com/ediprin/coffee-bean-detection.git',str(REPO)],check=True,cwd=WORK)
subprocess.run([sys.executable,'-m','pip','install','-q','ultralytics==8.4.96'],check=True,cwd=WORK)
subprocess.run([sys.executable,'-m','pip','install','-q','--no-deps','-e',str(REPO)],check=True,cwd=WORK)
sys.path.insert(0,str(REPO/'src')); importlib.invalidate_caches(); os.chdir(REPO)
import torch
if not torch.cuda.is_available(): raise RuntimeError('Aktifkan GPU Colab sebelum training')
print('COMMIT:',subprocess.check_output(['git','rev-parse','HEAD'],cwd=REPO,text=True).strip())
print('GPU:',torch.cuda.get_device_name(0))

In [ ]:
DATA=WORK/'defectoscafeverde-development-v1'
if DATA.exists(): shutil.rmtree(DATA)
DATA.mkdir(parents=True)
prefix='grouped-physical-v1/'; allowed_files={'data.yaml','grouped_audit.json','grouped_summary.json'}
with tarfile.open(ARCHIVE,'r') as bundle:
    for member in bundle.getmembers():
        name=member.name.replace('\\','/')
        if not name.startswith(prefix): continue
        relative=name[len(prefix):].strip('/')
        if not relative: continue
        first=relative.split('/',1)[0]
        if not (first in {'train','val'} or relative in allowed_files): continue
        target=(DATA/relative).resolve()
        if DATA.resolve() not in target.parents and target!=DATA.resolve(): raise RuntimeError('Unsafe archive path')
        if member.isdir(): target.mkdir(parents=True,exist_ok=True); continue
        target.parent.mkdir(parents=True,exist_ok=True); source=bundle.extractfile(member)
        if source is None: raise RuntimeError(f'Gagal membaca {member.name}')
        with source, target.open('wb') as out: shutil.copyfileobj(source,out)
if (DATA/'test').exists(): raise RuntimeError('TEST TEREXPOSE - STOP')
import yaml
payload=yaml.safe_load((DATA/'data.yaml').read_text()); payload.pop('test',None)
payload['path']=str(DATA); payload['train']='train/images'; payload['val']='val/images'
(DATA/'data.yaml').write_text(yaml.safe_dump(payload,sort_keys=False),encoding='utf-8')
AUDIT=DATA/'grouped_audit.json'
print('TRAIN:',len(list((DATA/'train/images').glob('*'))),'VAL:',len(list((DATA/'val/images').glob('*'))),'TEST:',(DATA/'test').exists())

In [ ]:
from ultralytics import YOLO
_=YOLO('yolo26n.pt'); PRETRAINED=(REPO/'yolo26n.pt').resolve()
OUT=PROJECT/'experiments/defectoscafeverde-ratf1-v1'; OUT.mkdir(parents=True,exist_ok=True)
d0=json.loads(D0.read_text())
if d0.get('protocol')!='defectoscafeverde-grouped-dcwcf-direct-seed42-v1' or d0.get('arm')!='D0DIRECT' or d0.get('test_images_accessed') is not False: raise RuntimeError('D0DIRECT reference tidak valid')
from coffee_detector.experiments.run_defectoscafeverde_ratf import run_static_preflight
from coffee_detector.experiments.run_defectoscafeverde_af2_direct import validate_development_dataset
contract=validate_development_dataset(DATA,AUDIT)
STATIC=OUT/'static_preflight_RATF1_notebook.json'
static=run_static_preflight(PRETRAINED,STATIC,seed=42)
print('D0 REFERENCE:',d0['metrics'])
print('PARAMETERS:',{'native':static['native_parameters'],'candidate':static['candidate_parameters'],'added':static['candidate_added_parameters']})
print('ORTHOGONALITY ERROR:',static['orthogonality_relative_error'])
print('DATA GATES:',contract['gates']); print('STATIC GATES:',static['gates']); print('DECISION:',static['decision'])
assert static['decision']=='PASS','STOP: static audit gagal; training tidak dijalankan.'

In [ ]:
LOG=OUT/f'{ARM}_seed42_run.log'
command=[sys.executable,'-u','-m','coffee_detector.experiments.run_defectoscafeverde_ratf','--data-root',str(DATA),'--grouped-audit',str(AUDIT),'--pretrained-checkpoint',str(PRETRAINED),'--output-root',str(OUT),'--seed','42','--device','0','--authorize-training']
print('START/RESUME:',ARM,'| log=',LOG,flush=True)
with LOG.open('a',encoding='utf-8') as stream: process=subprocess.Popen(command,cwd=REPO,stdout=stream,stderr=subprocess.STDOUT)
last=-1
while process.poll() is None:
    results=OUT/ARM/f'{ARM}_seed42'/'results.csv'
    epochs=sum(1 for _ in csv.DictReader(results.open(encoding='utf-8'))) if results.is_file() else 0
    if epochs!=last: print(f'{ARM}: {epochs}/50 epoch tercatat',flush=True); last=epochs
    time.sleep(120)
if process.returncode:
    print('\n'.join(LOG.read_text(errors='replace').splitlines()[-120:])); raise RuntimeError(f'{ARM} gagal: {process.returncode}')
RESULT=OUT/'val_reports'/f'{ARM}_seed42_result.json'
print(json.dumps(json.loads(RESULT.read_text()),indent=2,ensure_ascii=False))
print('last.pt tersimpan di Drive setiap epoch. Test tidak diekstrak.')

In [ ]:
from coffee_detector.experiments.run_defectoscafeverde_ratf import build_decision
decision=build_decision(D0,RESULT,OUT/'defectoscafeverde_ratf1_seed42_decision.json')
print('VALUES:',decision['values'])
print('DELTAS:',decision['deltas'])
print('HARD CLASS DELTAS:',decision['hard_class_deltas'])
print('PAIRED:',decision['paired'])
print('SCREEN:',decision['screen'])
print('TEST:',decision['test_opened'])
print('Kirim VALUES, DELTAS, HARD CLASS DELTAS, PAIRED, dan SCREEN. Jangan membuka test.')